In [ ]:
#| default_exp datasources.sitarweb
%load_ext autoreload
%autoreload 2

import sys,os
from pathlib import Path

In [ ]:
# Insert in Path Project Directory
sys.path.insert(0, str(Path().cwd().parent))
os.chdir(Path.cwd().parent / 'extracao')

# SITARWEB
> Módulo para encapsular a extração e processamento das bases de dados do SITARWEB

In [ ]:
#| export
import os
import sys
from decimal import Decimal, getcontext

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from fastcore.foundation import GetAttr

from extracao.constants import (
    COLS_STEL,
    COLS_RADCOM,
    SQL_RADCOM,
    SQL_STEL,
)

from extracao.datasources.base import Base
from extracao.datasources.connectors import SQLServer

In [ ]:
#| export
getcontext().prec = 5
load_dotenv(find_dotenv(), override=True)

True

In [ ]:
#| hide: true
#| eval:false
__file__ = Path.cwd().parent / 'extracao' / 'datasources.py'

In [ ]:
#| export
SQLSERVER_PARAMS = dict(
    driver=os.environ.get("SQL_DRIVER"),
    server=os.environ.get("SQL_SERVER"),
    database=os.environ.get("SQL_DATABASE"),
    trusted_conn=True,
    mult_results=True,
    encrypt=False,
    timeout=int(os.environ.get("SQL_TIMEOUT")),
)

if sys.platform in ("linux", "darwin", "cygwin"):
    SQLSERVER_PARAMS.update(
        {
            "trusted_conn": False,
            "mult_results": False,
            "username": os.environ.get("USERNAME"),
            "password": os.environ.get("PASSWORD"),
        }
    )


class Sitarweb(Base, GetAttr):
    """Common logic from the SITARWEB SQÇ Server Database"""

    def __init__(self, sql_params: dict = SQLSERVER_PARAMS, read_cache: bool = False):
        self.read_cache = read_cache
        self.default = SQLServer(sql_params)

    @property
    def columns(self):
        return COLS_STEL

    @property
    def query(self):
        raise NotImplementedError("Subclasses devem implementar a propriedade 'query'")

    def extraction(self):
        """This method returns a DataFrame with the results of the query"""
        if self.read_cache:
            df = self._read(f"{self.stem}_raw", "numpy_nullable")
        else:
            df = pd.read_sql_query(self.query, self.connect(), dtype="string")
        df["Log"] = "[]"
        return df

In [ ]:
#| export
class Radcom(Sitarweb):
    """Class for extracting data from SITARWEB refering to the"""

    def __init__(self, sql_params: dict = SQLSERVER_PARAMS, read_cache: bool = False):
        super().__init__(sql_params, read_cache)

    @property
    def query(self):
        return SQL_RADCOM

    @property
    def columns(self):
        return COLS_RADCOM


    @property
    def stem(self):
        return "radcom"

    def _format(
        self,
        df: pd.DataFrame,  # DataFrame com o resultantes do banco de dados
    ) -> pd.DataFrame:  # DataFrame formatado
        """Formata, limpa e padroniza os dados provenientes da query no banco"""
        df["Entidade"] = df["Entidade"].astype("string", copy=False).str.strip()
        df["Serviço"] = "231"
        df["Classe_Emissão"] = pd.NA
        df["Largura_Emissão(kHz)"] = "256"
        df["Validade_RF"] = pd.NA
        df["Status"] = "RADCOM"
        df["Fonte"] = "SRD"
        df["Multiplicidade"] = "1"
        df["Fase"] = df["Fase"].astype("string", copy=False)
        df["Situação"] = df["Situação"].astype("string", copy=False)

        a = df.Situação.isna()
        df.loc[a, "Classe"] = df.loc[a, "Fase"]
        # processing = 'Classe=Fase. Situação Nula. Colunas Fase e Situação descartadas'
        # Base.register_log(df, processing, row_filter=a)

        df.loc[~a, "Classe"] = df.loc[~a, "Fase"] + "-" + df.loc[~a, "Situação"]
        # processing = 'Classe=Fase-Situação. Colunas Fase e Situação descartadas'
        # Base.register_log(df, processing, row_filter=~a)
        df.drop(["Fase", "Situação"], axis=1, inplace=True)

        df["Frequência"] = pd.to_numeric(df["Frequência"], errors="coerce").astype(
            "float"
        )
        discarded = df[df.Frequência.isna()]
        if not discarded.empty:
            processing = "Coluna Frequência com valores nulos"
            Base.register_log(discarded, processing)
            self.append2discarded(discarded)
        df.dropna(subset=["Frequência"], inplace=True)
        return df.loc[:, self.columns]


In [ ]:
#| export
class Stel(Sitarweb):
    def __init__(self, sql_params: dict = SQLSERVER_PARAMS, read_cache: bool = False):
        super().__init__(sql_params, read_cache)

    @property
    def query(self):
        return SQL_STEL

    @property
    def stem(self):
        return "stel"

    def _format(
        self,
        df: pd.DataFrame,  # DataFrame com o resultantes do banco de dados
    ) -> pd.DataFrame:  # DataFrame formatado
        """Formata, limpa e padroniza os dados provenientes da query no banco"""
        df["Status"] = "L"
        df["Entidade"] = df.Entidade.astype("string", copy=False).str.strip()
        df["Fonte"] = "STEL"
        df["Largura_Emissão"] = df["Largura_Emissão"].astype("string", copy=False)
        df.loc[:, ["Largura_Emissão(kHz)", "_"]] = (
            df.Largura_Emissão.fillna("").apply(self.parse_bw).tolist()
        )
        df.drop(["Largura_Emissão", "_"], axis=1, inplace=True)
        df.loc[:, "Validade_RF"] = df.Validade_RF.astype(
            "string", copy=False
        ).str.slice(0, 10)
        df["Frequência"] = pd.to_numeric(df["Frequência"], errors="coerce").astype(
            "float"
        )
        df.loc[df.Unidade == "kHz", "Frequência"] = df.loc[
            df.Unidade == "kHz", "Frequência"
        ].apply(lambda x: float(Decimal(x) / Decimal(1000)))
        df.loc[df.Unidade == "GHz", "Frequência"] = df.loc[
            df.Unidade == "GHz", "Frequência"
        ].apply(lambda x: float(Decimal(x) * Decimal(1000)))
        df.drop("Unidade", axis=1, inplace=True)
        df["Multiplicidade"] = "1"
        return df.loc[:, self.columns]
